# 14 — Scoring (PageRank in SQL)

**AWS-side equivalent:** Aurora SQL PageRank (`notebooks/aws/14-scoring.ipynb`)
**GCP service:** AlloyDB for PostgreSQL — iterative PageRank in a recursive CTE; optional
BigQuery `ML.PAGE_RANK` sidebar for batch jobs.

Same composite formula as AWS:

$$\text{composite} = 0.40 \cdot \widehat{\text{expertise}} + 0.20 \cdot \widehat{\text{centrality}} + 0.30 \cdot \widehat{\text{pagerank}} + 0.10 \cdot \widehat{\text{betweenness}}$$

where $\widehat{x}$ is min-max scaled to $[0, 1]$. The result is written to
`people.composite_score`.

_As of 2026-05: AlloyDB AI does not ship a built-in `pagerank()` SQL function — implement it
in a recursive CTE. BigQuery has `ML.PAGE_RANK` (GA) which we show as an alternative for
large graphs._

## Step 1 — Load config and open an AlloyDB connection

In [ ]:
import json
import time
from pathlib import Path

config = json.loads(Path('config.json').read_text())
gcp = config.setdefault('gcp', {})
PROJECT_ID = gcp.get('project_id')
REGION = gcp.get('region', 'us-central1')
ALLOYDB_CLUSTER = gcp.get('alloydb_cluster', 'orgiq-alloydb')
ALLOYDB_INSTANCE = gcp.get('alloydb_instance', 'orgiq-alloydb-primary')
ALLOYDB_DB = gcp.get('alloydb_database', 'orgiq')
ALLOYDB_USER = gcp.get('alloydb_user', 'postgres')
BQ_DATASET = gcp.get('bq_scoring_dataset', 'orgiq_scoring')

assert PROJECT_ID, 'Run 01-setup-credentials.ipynb first.'
print(f'project={PROJECT_ID} region={REGION} alloydb_db={ALLOYDB_DB}')

In [ ]:
# Use the AlloyDB Python connector with IAM auth so we don't ship passwords.
from google.cloud.alloydb.connector import Connector, IPTypes
import sqlalchemy

INSTANCE_URI = (
    f'projects/{PROJECT_ID}/locations/{REGION}/clusters/{ALLOYDB_CLUSTER}/'
    f'instances/{ALLOYDB_INSTANCE}'
)

connector = Connector()

def _getconn():
    return connector.connect(
        INSTANCE_URI,
        'pg8000',
        user=ALLOYDB_USER,
        db=ALLOYDB_DB,
        enable_iam_auth=True,
        ip_type=IPTypes.PRIVATE,
    )

engine = sqlalchemy.create_engine('postgresql+pg8000://', creator=_getconn)
with engine.connect() as conn:
    n_people = conn.execute(sqlalchemy.text('SELECT count(*) FROM people')).scalar()
print(f'✓ AlloyDB reachable — {n_people} people in graph')

## Step 2 — Project a single weighted edge table

Three relations contribute to expertise propagation:

| Source table | Edge meaning | Default weight |
|---|---|---|
| `collaborations` | Person ↔ Person (already undirected) | `weight` column |
| `project_contributions` | shared project → person ↔ person | 0.5 × min(weight_a, weight_b) |
| `decision_participants` | shared decision → person ↔ person | 0.3 × min(weight_a, weight_b) |

We materialise these into a temp table `scoring_edges(src, dst, w)` keyed on `people.id` so
PageRank iteration touches a single dense relation.

In [ ]:
EDGE_PROJECTION = sqlalchemy.text("""
    DROP TABLE IF EXISTS scoring_edges;
    CREATE TEMP TABLE scoring_edges AS
    -- direct collaboration edges (mirror both directions)
    SELECT person_a_id AS src, person_b_id AS dst, weight::float8 AS w
      FROM collaborations
    UNION ALL
    SELECT person_b_id AS src, person_a_id AS dst, weight::float8 AS w
      FROM collaborations
    UNION ALL
    -- shared-project edges
    SELECT a.person_id AS src, b.person_id AS dst,
           0.5 * LEAST(a.weight, b.weight)::float8 AS w
      FROM project_contributions a
      JOIN project_contributions b USING (project_id)
     WHERE a.person_id <> b.person_id
    UNION ALL
    -- shared-decision edges
    SELECT a.person_id AS src, b.person_id AS dst,
           0.3 * LEAST(a.weight, b.weight)::float8 AS w
      FROM decision_participants a
      JOIN decision_participants b USING (decision_id)
     WHERE a.person_id <> b.person_id;

    -- aggregate parallel edges so each (src,dst) pair appears once
    DROP TABLE IF EXISTS scoring_edges_uniq;
    CREATE TEMP TABLE scoring_edges_uniq AS
    SELECT src, dst, SUM(w) AS w
      FROM scoring_edges
     GROUP BY src, dst;
""")

with engine.begin() as conn:
    conn.execute(EDGE_PROJECTION)
    n_edges = conn.execute(sqlalchemy.text('SELECT count(*) FROM scoring_edges_uniq')).scalar()

print(f'✓ scoring_edges_uniq materialised: {n_edges} unique edges')

## Step 3 — Iterative PageRank in a single SQL statement

AlloyDB recursive CTEs make the fixpoint iteration explicit. We run **20 iterations** with
**damping d = 0.85** and the standard formula:

$$PR_{t+1}(v) = \frac{1-d}{N} + d \sum_{u \to v} \frac{w_{u,v}}{W_u} \cdot PR_t(u)$$

where $W_u = \sum_{u \to *} w$ is the weighted out-degree.

The CTE seeds every node at $1/N$ and stops after `MAX_ITER` rounds. We then keep only the
final iteration's row per node and persist scores back to `people.pagerank_score`.

In [ ]:
MAX_ITER = 20
DAMPING = 0.85

PR_SQL = sqlalchemy.text(f"""
    WITH RECURSIVE
      params AS (
        SELECT {DAMPING}::float8 AS d,
               (SELECT count(*)::float8 FROM people) AS n,
               {MAX_ITER}::int AS max_iter
      ),
      out_w AS (
        SELECT src, SUM(w) AS w_total
          FROM scoring_edges_uniq
         GROUP BY src
      ),
      seed AS (
        SELECT p.id AS node_id, 1.0 / params.n AS pr, 0 AS iter
          FROM people p CROSS JOIN params
      ),
      iter(node_id, pr, iter) AS (
        SELECT node_id, pr, iter FROM seed
        UNION ALL
        SELECT p.id,
               (1.0 - params.d) / params.n
               + params.d * COALESCE(
                   SUM(e.w / out_w.w_total * prev.pr),
                   0.0),
               prev_iter.iter + 1
          FROM people p
          CROSS JOIN params
          CROSS JOIN (SELECT MAX(iter) AS iter FROM iter) prev_iter
          LEFT JOIN scoring_edges_uniq e ON e.dst = p.id
          LEFT JOIN out_w ON out_w.src = e.src
          LEFT JOIN iter prev ON prev.node_id = e.src AND prev.iter = prev_iter.iter
         WHERE prev_iter.iter < params.max_iter
         GROUP BY p.id, params.d, params.n, prev_iter.iter
      ),
      final AS (
        SELECT DISTINCT ON (node_id) node_id, pr
          FROM iter
         ORDER BY node_id, iter DESC
      )
    UPDATE people p
       SET pagerank_score = final.pr,
           scored_at = NOW()
      FROM final
     WHERE p.id = final.node_id
    RETURNING p.id, p.name, p.pagerank_score;
""")

t0 = time.time()
with engine.begin() as conn:
    rows = conn.execute(PR_SQL).fetchall()
elapsed = time.time() - t0

rows_sorted = sorted(rows, key=lambda r: r[2], reverse=True)
print(f'✓ PageRank updated for {len(rows_sorted)} people ({elapsed:.2f}s, {MAX_ITER} iters)')
print()
print(f"{'Rank':<5} {'Name':<28} {'PageRank':<14}")
print('-' * 50)
for i, (pid, name, pr) in enumerate(rows_sorted[:10], 1):
    print(f'{i:<5} {name:<28} {float(pr):.8f}')

## Step 4 — Expertise score (sum of `person_skills.weight`)

Direct skill weight is the simplest term in the composite — and the most legible to users.

In [ ]:
EXPERTISE_SQL = sqlalchemy.text("""
    UPDATE people p
       SET expertise_score = sub.score
      FROM (
        SELECT ps.person_id AS id, SUM(ps.weight)::float8 AS score
          FROM person_skills ps
         GROUP BY ps.person_id
      ) sub
     WHERE p.id = sub.id;
""")
with engine.begin() as conn:
    conn.execute(EXPERTISE_SQL)
print('✓ expertise_score updated')

## Step 5 — Centrality and betweenness in Python (Brandes)

AlloyDB has no built-in betweenness, and a SQL-only Brandes would be brutal to read.
We pull edges into Python, run Brandes' BFS algorithm on the undirected projection, and
write both centrality (`degree`) and betweenness back. For graphs > 1k people, sample 1k
sources and rescale (same trick as AWS-side notebook).

Stored on `people` as ad-hoc columns via `ALTER TABLE ... ADD COLUMN IF NOT EXISTS`.

In [ ]:
from collections import defaultdict, deque
import random

with engine.connect() as conn:
    person_rows = conn.execute(sqlalchemy.text('SELECT id FROM people')).fetchall()
    edge_rows = conn.execute(sqlalchemy.text('SELECT src, dst, w FROM scoring_edges_uniq')).fetchall()

person_ids = [r[0] for r in person_rows]
pid_set = set(person_ids)

adj = defaultdict(set)
for src, dst, _w in edge_rows:
    if src in pid_set and dst in pid_set:
        adj[src].add(dst)
        adj[dst].add(src)

centrality = {pid: len(adj[pid]) for pid in person_ids}

# Brandes' betweenness on the undirected projection.
n = len(person_ids)
if n > 1000:
    sources = random.sample(person_ids, 1000)
    scale = n / 1000.0
else:
    sources = person_ids
    scale = 1.0

betweenness = defaultdict(float)
for s in sources:
    stack = []
    pred = defaultdict(list)
    sigma = defaultdict(float)
    dist = defaultdict(lambda: -1)
    sigma[s] = 1.0
    dist[s] = 0
    queue = deque([s])
    while queue:
        v = queue.popleft()
        stack.append(v)
        for w in adj[v]:
            if dist[w] < 0:
                queue.append(w)
                dist[w] = dist[v] + 1
            if dist[w] == dist[v] + 1:
                sigma[w] += sigma[v]
                pred[w].append(v)
    delta = defaultdict(float)
    while stack:
        w = stack.pop()
        for v in pred[w]:
            delta[v] += (sigma[v] / sigma[w]) * (1.0 + delta[w])
        if w != s:
            betweenness[w] += delta[w]

norm = 1.0 / max(1, (n - 1) * (n - 2) / 2.0)
betweenness = {pid: betweenness[pid] * norm * scale for pid in person_ids}

print(f'✓ centrality + betweenness computed for {n} people')

In [ ]:
with engine.begin() as conn:
    conn.execute(sqlalchemy.text("""
        ALTER TABLE people
          ADD COLUMN IF NOT EXISTS centrality_score NUMERIC,
          ADD COLUMN IF NOT EXISTS betweenness_score NUMERIC;
    """))
    conn.execute(
        sqlalchemy.text(
            'UPDATE people SET centrality_score = :c, betweenness_score = :b WHERE id = :id'
        ),
        [
            {'id': pid, 'c': float(centrality[pid]), 'b': float(betweenness[pid])}
            for pid in person_ids
        ],
    )
print('✓ centrality_score + betweenness_score written')

## Step 6 — Composite score (min-max normalised weighted sum)

Same weights as the AWS notebook so the demo is comparable across clouds.

In [ ]:
COMPOSITE_SQL = sqlalchemy.text("""
    WITH bounds AS (
      SELECT
        MIN(expertise_score)    AS e_lo, MAX(expertise_score)    AS e_hi,
        MIN(centrality_score)   AS c_lo, MAX(centrality_score)   AS c_hi,
        MIN(pagerank_score)     AS p_lo, MAX(pagerank_score)     AS p_hi,
        MIN(betweenness_score)  AS b_lo, MAX(betweenness_score)  AS b_hi
      FROM people
    )
    UPDATE people p
       SET composite_score =
           0.40 * COALESCE((p.expertise_score   - b.e_lo) / NULLIF(b.e_hi - b.e_lo, 0), 0)
         + 0.20 * COALESCE((p.centrality_score  - b.c_lo) / NULLIF(b.c_hi - b.c_lo, 0), 0)
         + 0.30 * COALESCE((p.pagerank_score    - b.p_lo) / NULLIF(b.p_hi - b.p_lo, 0), 0)
         + 0.10 * COALESCE((p.betweenness_score - b.b_lo) / NULLIF(b.b_hi - b.b_lo, 0), 0),
           scored_at = NOW()
      FROM bounds b;
""")

LEADERBOARD_SQL = sqlalchemy.text("""
    SELECT name, composite_score, expertise_score, centrality_score,
           pagerank_score, betweenness_score
      FROM people
     ORDER BY composite_score DESC NULLS LAST
     LIMIT 15
""")

with engine.begin() as conn:
    conn.execute(COMPOSITE_SQL)
    board = conn.execute(LEADERBOARD_SQL).fetchall()

print(f"{'Rank':<5} {'Name':<24} {'Composite':<10} {'Exp':<8} {'Cent':<8} {'PR':<12} {'Betw':<12}")
print('-' * 80)
for i, row in enumerate(board, 1):
    name, comp, exp, cent, pr, betw = row
    print(f'{i:<5} {name:<24} {float(comp):<10.4f} '
          f'{float(exp or 0):<8.3f} {float(cent or 0):<8.1f} '
          f'{float(pr or 0):<12.6f} {float(betw or 0):<12.6f}')

## Step 7 — Alternative: BigQuery `ML.PAGE_RANK`

For graphs > 100k nodes the recursive CTE will choke. Vertex offers `ML.PAGE_RANK` in
BigQuery — fully parallel and free for graphs that fit in the on-demand tier.

Run this cell only if you've replicated `scoring_edges_uniq` into a BigQuery dataset (e.g.
via BQ Federation on AlloyDB or a Dataform/scheduled export job).

_As of 2026-05: `ML.PAGE_RANK` takes a `(node, neighbours)` shape and a `damping_factor`
option._

In [ ]:
# Sidebar — comment-only when running the AlloyDB-only path.
from google.cloud import bigquery

BQ_PR_QUERY = f"""
  DECLARE damping FLOAT64 DEFAULT 0.85;
  CREATE OR REPLACE TABLE `{PROJECT_ID}.{BQ_DATASET}.pagerank` AS
  SELECT node, pagerank
    FROM ML.PAGE_RANK(
      (
        SELECT src AS node, ARRAY_AGG(STRUCT(dst AS neighbour, w AS weight)) AS neighbours
          FROM `{PROJECT_ID}.{BQ_DATASET}.scoring_edges_uniq`
         GROUP BY src
      ),
      STRUCT(damping AS damping_factor, 20 AS max_iterations)
    );
"""

RUN_BIGQUERY_PAGERANK = False  # flip to True after exporting scoring_edges_uniq into BQ
if RUN_BIGQUERY_PAGERANK:
    bq = bigquery.Client(project=PROJECT_ID)
    job = bq.query(BQ_PR_QUERY)
    job.result()
    print(f'✓ BigQuery ML.PAGE_RANK written to {PROJECT_ID}.{BQ_DATASET}.pagerank')
else:
    print('skipped — flip RUN_BIGQUERY_PAGERANK to True after exporting the edge table.')
    print(BQ_PR_QUERY)

## Step 8 — Snapshot timestamp into config.json

In [ ]:
import datetime

snapshot_ts = int(time.time())
snapshot_iso = datetime.datetime.fromtimestamp(
    snapshot_ts, tz=datetime.timezone.utc
).isoformat()

config = json.loads(Path('config.json').read_text())
config.setdefault('gcp', {})['scoring_snapshot_ts'] = snapshot_ts
config['gcp']['scoring_snapshot_iso'] = snapshot_iso
config['gcp']['scoring_person_count'] = len(person_ids)
Path('config.json').write_text(json.dumps(config, indent=2))

print(f'✓ snapshot_ts={snapshot_ts} ({snapshot_iso})')
print('Next: 15-deploy-agent-engine.ipynb — deploy ADK agents to Agent Engine')